# Backpropagation

- Thursdays, 3:30–6:00 PM · ICC 103
- Weeks 3 and 4 of 14 · Sep 10 and Sep 17
- **Quiz 2** at the end of Sep 10; **Quiz 3** at the end of Sep 17

## Agenda

1. **Why backpropagation exists**: what Lab 2 cost, priced at scale
2. **The chain rule**, in the form deep learning actually uses it
3. **Computational graphs**: the forward pass, recorded
4. **Reverse-mode autodiff**, worked by hand
5. **From a scalar to a layer**, and why nobody stores a Jacobian

Spotlights first, then the quiz in the last 20 minutes.

::: {.callout-note}
## PyTorch comes after this deck
Everything here is the chain rule and NumPy. Once you can do it by hand,
`.backward()` is a one-line replacement, and that is where Week 5 starts.
:::

## Lab 2 recap

You trained a $1 \to 16 \to 1$ tanh network on a noisy sine with **no
backpropagation**. Finite differences, 400 steps of gradient descent:

| | |
|---|---|
| Parameters | 49 |
| Forward passes per gradient | 50 |
| Total forward passes | 20,001 |
| Cost, start to end | 1.3256 → 0.0190 |
| A straight line, for comparison | 0.1736 |
| The noise floor | 0.0113 |

The fit was real. It beat a straight line by an order of magnitude and landed
near the noise floor. No library on earth trains a network this way.

## Cost of one gradient step {.smaller}

[![](images/fig-cost-scaling.png){width=1150 .dw95}](images/fig-cost-scaling.png){target="_blank" .zoom}

Left is measured on this laptop: the gap widens from 150x to over 1,600x across
that range. Right is not measured, because it does not need to be. Finite
differences take **exactly** $n+1$ forward passes for $n$ parameters. At one
millisecond each, ResNet-50 needs 7 hours and GPT-3 needs 5.5 years, for **one**
step of gradient descent.

## Limits of analytical solutions

Week 1 found the least-squares solution in closed form. That does not generalize:

1. **Scale.** 175 billion parameters is 175 billion coupled equations
2. **Nonlinearity.** $\tanh$, ReLU and softmax compose into transcendental
   equations with no algebraic solution
3. **Non-convexity.** Many local minima and many saddle points, so "the"
   solution is not a well-posed request
4. **No closed form.** Setting the gradient to zero gives you a system nobody
   can solve symbolically

So we iterate. Iterating needs a gradient at the current parameters, and it
needs one **per step**. The gradient is the thing that has to be cheap.

## Three ways to get a gradient

| | Exact? | Cost for $n$ parameters | Works on a network? |
|---|---|---|---|
| Symbolic, by hand | yes | your whole afternoon, once | only for tiny ones |
| Finite differences | **no**, approximate | $n+1$ forward passes | yes, and too slowly |
| Reverse-mode autodiff | **yes**, to floating point | 1 forward + 1 backward | this is what we use |

Finite differences are worse on both axes. They are approximate **and** slow.
The step size $\varepsilon$ has to be small enough to be accurate and large
enough to survive floating-point cancellation, and those two demands fight.

Autodiff has no $\varepsilon$. It is exact arithmetic on the derivative rules
you already know.

## Automatic differentiation

**Automatic differentiation** computes partial derivatives of an arbitrary
computer program by applying the chain rule to every elementary operation the
program ran.

It works because every line of a forward pass is one of a short list:

- **Operations:** add, subtract, multiply, divide
- **Functions:** $\exp$, $\log$, $\sin$, $\tanh$, $\max$

Each has a derivative you learned in Calc I. Autodiff records which ones ran,
in what order, on what values, and then walks that record backward.

PyTorch, TensorFlow and JAX are all autodiff engines with a neural-network
library attached.

## Backpropagation

**Backpropagation** is reverse-mode autodiff applied to a neural network. It
answers, for every weight and every bias in the model:

$$\frac{\partial \mathcal{L}}{\partial w} = \; ? \qquad
  \frac{\partial \mathcal{L}}{\partial b} = \; ?$$

after **one** forward pass and **one** backward pass, regardless of how many
parameters there are.

::: {.callout-note}
## The name is narrower than the idea
Autodiff is the general procedure and predates deep learning. Backpropagation
is its application to networks. In practice people use the words
interchangeably, and you should know which one is the superset.
:::

## Loss and cost {.smaller}

These get used interchangeably. They are not the same thing, and today the
distinction matters because the backward pass starts at one of them.

:::: {.columns}
::: {.column width="48%"}
**Loss** $\ell_i$: the error on a **single** example.

$$\ell_i = (y_i - \hat{y}_i)^2$$

$$\ell_i = -\big(y_i \log \hat{y}_i + (1-y_i)\log(1-\hat{y}_i)\big)$$
:::
::: {.column width="48%"}
**Cost** $\mathcal{L}$: the reduction of those losses over a batch of $N$.

$$\mathcal{L} = \frac{1}{N}\sum_{i=1}^{N} \ell_i$$

The "mean" in *mean* squared error is what turns a loss into a cost.
:::
::::

The optimizer only ever sees a **scalar**. `loss.backward()` in PyTorch is
called on the cost, and that scalar is why reverse mode is the right choice.

# The chain rule

- One rule, applied a few million times
- The multivariable version is the one that matters

## Chain rule for composite functions

A network is a composition of simple functions, so the chain rule is the entire
mathematical content of backpropagation. There is no limit to the depth of the
composition:

$$\frac{d}{dx}\big[f(g(h(x)))\big]
  = f'(g(h(x)))\; g'(h(x))\; h'(x)$$

Take $f(x) = e^x$, $g(x) = \sin x$, $h(x) = x^2$, so
$f(g(h(x))) = e^{\sin(x^2)}$:

$$\frac{d}{dx}\big[e^{\sin(x^2)}\big]
  = e^{\sin(x^2)} \cdot \frac{d}{dx}\big[\sin(x^2)\big]
  = e^{\sin(x^2)} \cos(x^2) \cdot \frac{d}{dx}\big[x^2\big]
  = 2x\, e^{\sin(x^2)} \cos(x^2)$$

Read it right to left and you have just done a backward pass on a three-node
graph.

## Multivariable chain rule

Networks have thousands to billions of parameters, and a parameter early in the
network reaches the cost through **many** downstream paths. The single-variable
rule is not enough.

Let $z = f(x, y)$ where $x = g(w, b)$ and $y = h(w, b)$. Then

$$\frac{\partial z}{\partial w}
  = \frac{\partial z}{\partial x}\frac{\partial x}{\partial w}
  + \frac{\partial z}{\partial y}\frac{\partial y}{\partial w}$$

$$\frac{\partial z}{\partial b}
  = \frac{\partial z}{\partial x}\frac{\partial x}{\partial b}
  + \frac{\partial z}{\partial y}\frac{\partial y}{\partial b}$$

Think of $x$ and $y$ as two hidden units that both feed a unit downstream.
$w$ affects the result through both, so **both paths are counted, and they add.**

## Multivariable chain rule, worked

Let $z = e^{xy}$ where $x = 2w + b$ and $y = \sin w + \cos 2b$.

The local derivatives are
$\partial z/\partial x = y e^{xy}$, $\partial z/\partial y = x e^{xy}$,
$\partial x/\partial w = 2$, $\partial y/\partial w = \cos w$.

$$\frac{\partial z}{\partial w}
  = \underbrace{y e^{xy}}_{\text{path through } x} \cdot 2
  + \underbrace{x e^{xy}}_{\text{path through } y} \cdot \cos w$$

$$\frac{\partial z}{\partial b}
  = y e^{xy} \cdot 1
  + x e^{xy} \cdot (-2 \sin 2b)$$

Nothing here is hard. It is bookkeeping. Autodiff exists because the bookkeeping
does not stay manageable past about six nodes.

## Chain rule along a network path {.smaller}

[![](images/2024-03-12-17-07-44.png){width=735 .dw70}](images/2024-03-12-17-07-44.png){target="_blank" .zoom}

The bottom rows are the part to study: each factor written once as a **formula**
and once as its **value**, with $\partial L / \partial w$ the product of the
row, $(1)(2)(1)(0.187)(-16) \approx -6$.

## Generalized chain rule

The statement with no ceiling on the number of functions or variables, which is
the shape a real network takes.

Let $z = f(x_1, x_2, \ldots, x_m)$ be differentiable in $m$ variables, and let
each $x_i = x_i(t_1, t_2, \ldots, t_n)$ be differentiable in $n$ variables.
Then for any $j$:

$$\frac{\partial z}{\partial t_j}
  = \frac{\partial z}{\partial x_1}\frac{\partial x_1}{\partial t_j}
  + \frac{\partial z}{\partial x_2}\frac{\partial x_2}{\partial t_j}
  + \cdots
  + \frac{\partial z}{\partial x_m}\frac{\partial x_m}{\partial t_j}$$

Read $z$ as the cost, $t_j$ as one weight, and the $x_i$ as every activation
that weight touches. That sum is the backward pass.

## Gradient notation

The **gradient** of a function collects all of its partial derivatives into one
object, written $\nabla f$.

For $z = f(x_1, \ldots, x_m)$:

$$\nabla_{\mathbf{x}} f =
  \begin{bmatrix}
  \partial f / \partial x_1 \\
  \partial f / \partial x_2 \\
  \vdots \\
  \partial f / \partial x_m
  \end{bmatrix}$$

In deep learning we want the gradient of the cost with respect to the
**parameters**, written $\nabla_{\theta} \mathcal{L}$, where $\theta$ is the
whole parameter set. Lab 2 called that flat vector `theta` and it had 49 entries.

## Gradient of the cost with respect to a weight matrix {.smaller}

[![](images/fig-gradient-shapes.png){width=1050 .dw70}](images/fig-gradient-shapes.png){target="_blank" .zoom}

For $\mathbf{W}$ of shape $(m, n)$, the gradient
$\nabla_{\mathbf{W}} \mathcal{L}$ is also $(m, n)$:

$$\nabla_{\mathbf{W}} \mathcal{L} =
  \begin{bmatrix}
  \partial \mathcal{L}/\partial w_{1,1} & \cdots & \partial \mathcal{L}/\partial w_{1,n} \\
  \vdots & \ddots & \vdots \\
  \partial \mathcal{L}/\partial w_{m,1} & \cdots & \partial \mathcal{L}/\partial w_{m,n}
  \end{bmatrix}$$

This is worth pausing on, because it is what makes `theta = theta - lr * grad`
a legal line of code.

# Computational graphs

- Nodes are values, edges are operations
- The forward pass builds the graph; the backward pass walks it

## Computational graphs as directed acyclic graphs

[![](images/computational-graph-sigmoid.png){width=735 .dw70}](images/computational-graph-sigmoid.png){target="_blank" .zoom}

A network's computational graph is a **directed acyclic graph**:

- Each node is a value that some operation produced
- Each edge is an operation with a known derivative
- **Acyclic** means no cycles, which is what makes a single backward sweep
  well-defined
- Nodes can be as primitive or as coarse as you like. $\sigma(z)$ can be four
  nodes or one

## Graph of an expression {.smaller}

:::: {.columns}
::: {.column width="42%"}
[![](images/computational-graph-2.png){width=420 .dw40}](images/computational-graph-2.png){target="_blank" .zoom}
:::
::: {.column width="58%"}
Before any numbers, the graph is just the structure of the expression. Reading
it left to right: $w$ and $x$ meet at a multiply, the result meets $v$ at an
add, and that meets $-1$ at a multiply.

Nothing here is specific to neural networks. Any expression built from
differentiable pieces has a graph like this, which is why autodiff works on
ordinary Python code and not only on models.
:::
::::

## Forward pass on a graph {.smaller}

[![](images/kahns-algorithm.png){width=714 .dw68}](images/kahns-algorithm.png){target="_blank" .zoom}

Three inputs and four operations. $A \times B = C$, then $D + C = F$, then
$F + A = G$, then $G \times B = H$. The **green** numbers are the forward
values; ignore the orange ones for now.

The forward pass does two things, and the second is the one people forget: it
computes the values, **and it records the operations and their inputs.** That
record is what the backward pass consumes, and it is why a forward pass under
`torch.no_grad()` uses less memory.

## The chain rule on a graph

The rule becomes local. At a node $v$ with parent $u$:

$$\frac{\partial \mathcal{L}}{\partial u}
  = \underbrace{\frac{\partial \mathcal{L}}{\partial v}}_{\text{arrives from the right}}
    \cdot
    \underbrace{\frac{\partial v}{\partial u}}_{\text{one derivative, computed here}}$$

Each node needs to know exactly two things:

1. The gradient handed to it by its children, called the **upstream gradient**
2. The derivative of its own operation, called the **local gradient**

Multiply them, pass the product to the parents. No node knows the size of the
network it is in, and none of them needs to.

## Local derivatives at a node

[![](images/backpropagation-node-types.png){width=945 .dw90}](images/backpropagation-node-types.png){target="_blank" .zoom}

- **Add.** $z = a + b$ has $\partial z/\partial a = 1$, so the upstream
  gradient is **copied** to every input
- **Multiply.** $z = ab$ has $\partial z/\partial a = b$, so the gradients
  **swap**: $a$ receives $b \cdot \partial \mathcal{L}/\partial z$
- **Max.** The gradient **routes** to whichever input won, and the other gets
  zero. ReLU is $\max(0, z)$, so a negative pre-activation gets nothing

Three rules cover most of a feedforward network. What does a **min** gate do?
What about concatenation?

## Backward pass on a graph {.smaller}

[![](images/kahns-algorithm.png){width=714 .dw68}](images/kahns-algorithm.png){target="_blank" .zoom}

Same graph, now read the **orange** numbers, which are the gradients of $H$
with respect to each edge. They are produced right to left, starting from
$\partial H / \partial H = 1$ at the output.

Every orange number is an upstream gradient times one local derivative. At
$H = G \times B$, for instance, $G$ receives $1 \cdot B = 3$ and $B$ receives
$1 \cdot G = 13$; the multiply node swaps its inputs.

## Gradients sum over paths

[![](images/fig-fork-adds.png){width=1050 .dw90}](images/fig-fork-adds.png){target="_blank" .zoom}

$A$ reaches the output through $C$ and through $G$, so the multivariable chain
rule says count both and add: $9 + 3 = 12$.

This is not a detail. It is why the backward pass **adds into** a node's stored
gradient rather than overwriting it, which is the `+=` in Lab 3's `Value` class.
It is also why PyTorch makes you call `zero_grad()`, as we will see next week.

## A sigmoid expanded into primitive nodes {.smaller}

[![](images/backpropagation-single-node-sigmoid.png){width=725 .dw68}](images/backpropagation-single-node-sigmoid.png){target="_blank" .zoom}

Written out, $\sigma(z) = 1/(1 + e^{-z})$ is four operations: negate, exponentiate,
add one, take the reciprocal. Each is a node with its own local derivative, and
backprop runs through all four.

## The same sigmoid as one node {.smaller}

[![](images/backpropagation-single-node-sigmoid-simplified.png){width=725 .dw60}](images/backpropagation-single-node-sigmoid-simplified.png){target="_blank" .zoom}

Because $\sigma$ has a derivative we know in closed form, those four nodes
collapse into one:

$$\sigma'(z) = \sigma(z)\big(1 - \sigma(z)\big)$$

Same gradient, one node instead of four. The forward pass already computed
$\sigma(z)$, so the backward pass gets the local gradient for free from a value
it has cached. This is the general pattern: autodiff engines trade memory for
speed by keeping forward values around.

## Execution order and topological sorting

A node cannot compute its gradient until **every** child has handed one over.
Look at $A$ in the last figure: process it too early and you get $-3$ instead of
$-12$.

So the engine needs an ordering of the nodes where no node comes before something
it depends on. That ordering is a **topological sort**, and it is not unique. For
the graph below, both of these are valid:

$$[0, 1, 2, 3, 4, 5, 6, 7] \qquad [1, 0, 4, 3, 7, 6, 5, 2]$$

[![](images/topological-sort.png){width=420 .dw40}](images/topological-sort.png){target="_blank" .zoom}

## Kahn's algorithm

Repeatedly take a node with no remaining edges into it, emit it, and delete its
edges. If edges are left over at the end, the graph had a cycle and there is no
valid order.

We will walk the next eight slides one at a time. Watch two things:

- the **queue** on the left, which holds every node that is ready to be emitted
- the **topological order** along the bottom, which fills left to right

We start at the output node and work back toward the inputs, so the order this
produces is exactly the order `loss.backward()` visits nodes in.

## Topological sort: the starting queue

[![](images/kahns-algorithm-00.png){width=880 .dw85}](images/kahns-algorithm-00.png){target="_blank" .zoom}

## Topological sort: step 1

[![](images/kahns-algorithm-01.png){width=880 .dw85}](images/kahns-algorithm-01.png){target="_blank" .zoom}

## Topological sort: step 2

[![](images/kahns-algorithm-02.png){width=880 .dw85}](images/kahns-algorithm-02.png){target="_blank" .zoom}

## Topological sort: step 3

[![](images/kahns-algorithm-03.png){width=880 .dw85}](images/kahns-algorithm-03.png){target="_blank" .zoom}

## Topological sort: step 4

[![](images/kahns-algorithm-04.png){width=880 .dw85}](images/kahns-algorithm-04.png){target="_blank" .zoom}

## Topological sort: step 5

[![](images/kahns-algorithm-05.png){width=880 .dw85}](images/kahns-algorithm-05.png){target="_blank" .zoom}

## Topological sort: step 6

[![](images/kahns-algorithm-06.png){width=880 .dw85}](images/kahns-algorithm-06.png){target="_blank" .zoom}

## Topological sort: the finished order

[![](images/kahns-algorithm-07.png){width=880 .dw85}](images/kahns-algorithm-07.png){target="_blank" .zoom}

## Topological sort, animated

[![](images/kahns-algorithm.gif){width=880 .dw85}](images/kahns-algorithm.gif){target="_blank" .zoom}

The same eight steps, run end to end.

## Kahn's algorithm in pseudocode {.smaller}

```
L <- empty list that will hold the sorted elements
S <- set of all nodes with no incoming edge

while S is not empty:
    remove a node n from S
    add n to L
    for each node m with an edge e from n to m:
        remove edge e from the graph
        if m has no other incoming edges:
            insert m into S

if the graph still has edges:
    return error          # at least one cycle
else:
    return L              # a topologically sorted order
```

You will not implement this. You should know that it exists, that it is why the
graph has to be acyclic, and that a recurrent network gets around the "acyclic"
requirement by unrolling in time, which is Week 9.

# Reverse-mode autodiff

- One scalar example, worked by hand
- Then the same example checked against a real autodiff engine

## Scalar regression example {.smaller}

Take $x = 3$, $w = 2$, $b = 4$, a ReLU, and squared error against $y = 12$:

$$z_1 = wx = 6, \quad
  z_2 = z_1 + b = 10, \quad
  a = \mathrm{ReLU}(z_2) = 10, \quad
  \ell = (a - 12)^2 = 4$$

Four numbers and four recorded operations. This is a single neuron, which is
exactly the object Week 2 built. Now walk it backward, starting from
$\partial \ell / \partial \ell = 1$:

$$\frac{\partial \ell}{\partial a} = 2(a - y) = -4,
  \quad
  \frac{\partial \ell}{\partial z_2} = -4 \cdot \mathbb{1}[z_2 > 0] = -4,
  \quad
  \frac{\partial \ell}{\partial w} = -4x = -12,
  \quad
  \frac{\partial \ell}{\partial b} = -4$$

ReLU passed the gradient through because $z_2 > 0$. Had $z_2$ been negative,
every gradient to its left would be exactly zero and this neuron would learn
nothing from this example.

## Scalar regression graph, both passes {.smaller}

[![](images/scalar-backpropagation-regression.png){width=945 .dw90}](images/scalar-backpropagation-regression.png){target="_blank" .zoom}

Green is the forward pass, orange the backward pass, and every node carries the
chain rule that produced its number. ReLU and MSE, with their derivatives, are
in the top right.

## Scalar example, verified against autograd

Two lines of setup, so the check is readable. `requires_grad=True` tells the
engine to record; `retain_grad()` asks it to keep the gradient at an
intermediate node, which it normally discards. **The API itself is next week**,
so read this as a second opinion on the numbers you just derived.

In [1]:
#| echo: true
import torch
import torch.nn as nn

x = torch.tensor([3.0], requires_grad=True)
w = torch.tensor([2.0], requires_grad=True)
b = torch.tensor([4.0], requires_grad=True)
y = torch.tensor([12.0])

z1 = w * x;                z1.retain_grad()
z2 = z1 + b;               z2.retain_grad()
a = torch.relu(z2);        a.retain_grad()
loss = nn.MSELoss()(a, y)

loss.backward()

print(f"forward    z1={z1.item():5.1f}  z2={z2.item():5.1f}  "
      f"a={a.item():5.1f}  loss={loss.item():5.1f}")
print()
for name, t in [("a", a), ("z2", z2), ("z1", z1), ("b", b), ("w", w), ("x", x)]:
    print(f"  dloss/d{name:<2} {t.grad.item():>7.1f}")

forward    z1=  6.0  z2= 10.0  a= 10.0  loss=  4.0

  dloss/da     -4.0
  dloss/dz2    -4.0
  dloss/dz1    -4.0
  dloss/db     -4.0
  dloss/dw    -12.0
  dloss/dx     -8.0


## Reading gradients out of `.grad`

It approximated nothing. It ran the chain rule you just ran, on the graph it
recorded during the forward pass, and every number matches the hand derivation.

- `w.grad = -12` is $\partial \ell / \partial w$. SGD will use it
- `b.grad = -4` is $\partial \ell / \partial b$. SGD will use it
- `x.grad = -8` is $\partial \ell / \partial x$. It gets computed on the way
  past and then discarded: the input is not a parameter, so it never enters a
  weight update, and nothing left in the backward pass depends on it

Finite differences would have needed one forward pass per scalar. This needed
one forward and one backward, for all of them at once.

::: {.callout-tip}
## Optimizing the input instead of the weights
Discarded during training does not mean it has no use. Adversarial examples, saliency
maps and style transfer all hold the weights fixed and optimize the **input**,
which makes $\partial \ell / \partial x$ the quantity they need. Same
machinery, different variable.
:::

## Scalar classification graph

[![](images/scalar-backpropagation-classification-simplified.png){width=714 .dw68}](images/scalar-backpropagation-classification-simplified.png){target="_blank" .zoom}

Same shape, different head: sigmoid instead of ReLU, binary cross-entropy
instead of squared error. Take $x = 2$, $w = 0.5$, $b = -2.1$, and $y = 0$.

$$z_2 = -1.1, \qquad
  \hat{y} = \sigma(z_2) = 0.2497, \qquad
  \ell = -\log(1 - \hat{y}) = 0.2873$$

## Scalar classification graph, both passes

[![](images/scalar-backpropagation-classification.png){width=1050 .dw95}](images/scalar-backpropagation-classification.png){target="_blank" .zoom}

The same expansion as the regression case, now with a sigmoid and binary cross
entropy. The chain is longer because the loss is written out as
$\hat{y} = 1/a_3$ rather than folded into one node.

## Derivative of binary cross-entropy {.smaller}

[![](images/2024-03-12-20-12-43.png){width=714 .dw68}](images/2024-03-12-20-12-43.png){target="_blank" .zoom}

Worth keeping for the two class cases at the middle of the slide: the same
formula gives $+1.33$ when $y = 0$ and $-4$ when $y = 1$, from the identical
prediction $\hat{y} = 0.25$. The sign is what tells the weight which way to
move.

## Classification example, verified against autograd

Watch $\partial \ell / \partial z_2$ in the output and compare it to
$\hat{y} - y$.

In [2]:
#| echo: true
x = torch.tensor([2.0], requires_grad=True)
w = torch.tensor([0.5], requires_grad=True)
b = torch.tensor([-2.1], requires_grad=True)
y = torch.tensor([0.0])

z1 = w * x;                    z1.retain_grad()
z2 = z1 + b;                   z2.retain_grad()
y_hat = torch.sigmoid(z2);     y_hat.retain_grad()
loss = nn.BCELoss()(y_hat, y)

loss.backward()

print(f"z2 = {z2.item():.4f}   y_hat = {y_hat.item():.4f}   "
      f"loss = {loss.item():.4f}")
print()
print(f"  dloss/dz2  {z2.grad.item():.4f}")
print(f"  y_hat - y  {(y_hat - y).item():.4f}   <- the same number")
print()
for name, t in [("w", w), ("b", b), ("x", x)]:
    print(f"  dloss/d{name}   {t.grad.item():>7.4f}")

z2 = -1.1000   y_hat = 0.2497   loss = 0.2873

  dloss/dz2  0.2497
  y_hat - y  0.2497   <- the same number

  dloss/dw    0.4995
  dloss/db    0.2497
  dloss/dx    0.1249


## Gradient of cross-entropy with a sigmoid output {.smaller}

Two derivatives stand between the loss and $z$. Watch the denominator of the
first one.

**1. Differentiate the loss.** With
$\ell = -\big(y\log\hat{y} + (1-y)\log(1-\hat{y})\big)$, put the two terms over
a common denominator:

$$\frac{\partial \ell}{\partial \hat{y}}
  = -\frac{y}{\hat{y}} + \frac{1-y}{1-\hat{y}}
  = \frac{-y(1-\hat{y}) + (1-y)\hat{y}}{\hat{y}(1-\hat{y})}
  = \frac{\hat{y} - y}{\hat{y}(1-\hat{y})}$$

**2. Differentiate the sigmoid.** This is the identity from Week 2:

$$\frac{\partial \hat{y}}{\partial z} = \sigma(z)\big(1 - \sigma(z)\big) = \hat{y}(1-\hat{y})$$

**3. Multiply.** Step 2 is exactly the denominator left over from step 1, so it
divides out:

$$\frac{\partial \ell}{\partial z}
  = \frac{\hat{y} - y}{\hat{y}(1-\hat{y})} \cdot \hat{y}(1-\hat{y})
  = \frac{(\hat{y} - y)\,\hat{y}(1-\hat{y})}{\hat{y}(1-\hat{y})}
  = \boxed{\hat{y} - y}$$

Check it against the cell above: $\hat{y} - y = 0.2497 - 0 = 0.2497$, which is
what autograd printed for $\partial \ell / \partial z_2$.

## Why sigmoid pairs with cross-entropy

The gradient entering the linear part is **the prediction minus the truth**.
Squared error on a linear output gives the same form, and it is the same algebra
Week 1 met in logistic regression. These pairings are the defaults because they
are the ones that cancel.

The cancellation also removes $\sigma'$ from the gradient, and that matters more
than the tidiness. Pair a sigmoid with **squared error** instead and $\sigma'$
survives, so a confidently wrong prediction barely corrects itself. With
$y = 0$ and the model insisting $\hat{y} = 0.999$:

| Output pairing | $\partial \ell / \partial z$ |
|---|---|
| sigmoid + cross-entropy | $0.999$ |
| sigmoid + squared error | $0.002$ |

Same prediction, same error, a gradient 500 times smaller. That is the reason
classification does not use squared error, and it is the vanishing-gradient
problem in miniature; we meet the full version shortly.

## Forward mode and reverse mode

[![](images/fig-forward-vs-reverse.png){width=1100 .dw85}](images/fig-forward-vs-reverse.png){target="_blank" .zoom}

Autodiff has two directions, and the choice is not stylistic.

- **Forward mode** propagates a derivative from one input forward. To get all
  $n$ parameter gradients you sweep $n$ times
- **Reverse mode** propagates a derivative from one output backward. One sweep
  gives you every input gradient

## Choosing forward or reverse mode

The cost of each mode is set by the **shape** of the function, not by the
architecture:

| | Sweeps needed | Good when |
|---|---|---|
| Forward mode | one per **input** | few inputs, many outputs |
| Reverse mode | one per **output** | many inputs, **one** output |

A neural network maps millions of parameters to a **single scalar cost**. That
is the best possible case for reverse mode and the worst possible case for
forward mode.

If the cost were a vector of a million values, forward mode would win and deep
learning would look very different. It is a scalar because we chose to reduce
the batch to a mean, which brings us back to loss versus cost.

# From a scalar to a layer

- The same three rules, now on matrices
- What gets stored, and what does not

## Backward pass of a linear layer

Week 2: a layer with $\mathbf{W}$ of shape `(units, inputs)` computes
$\mathbf{z} = \mathbf{W}\mathbf{x} + \mathbf{b}$.

Given $\partial \mathcal{L} / \partial \mathbf{z}$ arriving from the next
layer, the layer owes three things:

$$\frac{\partial \mathcal{L}}{\partial \mathbf{W}}
  = \frac{\partial \mathcal{L}}{\partial \mathbf{z}}\, \mathbf{x}^{\top},
  \qquad
  \frac{\partial \mathcal{L}}{\partial \mathbf{b}}
  = \frac{\partial \mathcal{L}}{\partial \mathbf{z}},
  \qquad
  \frac{\partial \mathcal{L}}{\partial \mathbf{x}}
  = \mathbf{W}^{\top} \frac{\partial \mathcal{L}}{\partial \mathbf{z}}$$

The first two are what the optimizer updates. The third is the upstream gradient
for the layer **behind** this one, which is what makes the pass propagate.

## Shape check on the layer gradients

The formulas are memorable because the shapes force them. A layer with 3 units
and 2 inputs, on a single example:

| Object | Shape | Why |
|---|---|---|
| $\mathbf{W}$ | (3, 2) | units by inputs |
| $\mathbf{x}$ | (2,) | what came in |
| $\partial \mathcal{L} / \partial \mathbf{z}$ | (3,) | one per unit |
| $\partial \mathcal{L} / \partial \mathbf{W}$ | (3, 2) | (3,) outer (2,) |
| $\partial \mathcal{L} / \partial \mathbf{x}$ | (2,) | (2, 3) times (3,) |

Only one arrangement of $\mathbf{W}$, $\mathbf{W}^{\top}$ and the incoming
gradient produces each required shape. **If the shapes work, the formula is
almost certainly right**, and that is the debugging trick to remember.

## Layer backward pass, verified against autograd {.codetight}

Lab 2's 2-3-2-1 network, same seed. The whole backward pass written out in
NumPy, then checked against `loss.backward()`. Source code for your reference;
the numbers are the point.

In [3]:
#| echo: true
import numpy as np

rng = np.random.default_rng(6600)
W1, b1 = rng.normal(0, 0.8, (3, 2)), np.zeros(3)
W2, b2 = rng.normal(0, 0.8, (2, 3)), np.zeros(2)
W3, b3 = rng.normal(0, 0.8, (1, 2)), np.zeros(1)
X, Y = rng.normal(0, 1, (5, 2)), rng.normal(0, 1, (5, 1))
N = X.shape[0]

# forward, keeping every activation because the backward pass needs them
z1 = X @ W1.T + b1;   a1 = np.tanh(z1)
z2 = a1 @ W2.T + b2;  a2 = np.tanh(z2)
z3 = a2 @ W3.T + b3
cost = float(np.mean((z3 - Y) ** 2))

# backward, right to left: seed, then (local gradient) x (upstream gradient)
dz3 = 2 * (z3 - Y) / N                     # d cost / d z3
gW3, gb3 = dz3.T @ a2, dz3.sum(0)          # this layer's parameters
dz2 = (dz3 @ W3) * (1 - a2 ** 2)           # through W3, then through tanh
gW2, gb2 = dz2.T @ a1, dz2.sum(0)
dz1 = (dz2 @ W2) * (1 - a1 ** 2)
gW1, gb1 = dz1.T @ X, dz1.sum(0)

# the same thing, asked of autograd
model = nn.Sequential(nn.Linear(2, 3), nn.Tanh(),
                      nn.Linear(3, 2), nn.Tanh(),
                      nn.Linear(2, 1)).double()
with torch.no_grad():
    for layer, (W, b) in zip([model[0], model[2], model[4]],
                             [(W1, b1), (W2, b2), (W3, b3)]):
        layer.weight.copy_(torch.from_numpy(W))
        layer.bias.copy_(torch.from_numpy(b))
torch_cost = nn.MSELoss()(model(torch.from_numpy(X)), torch.from_numpy(Y))
torch_cost.backward()

print(f"cost   by hand {cost:.6f}   autograd {float(torch_cost):.6f}\n")
print(f"{'':>4}{'shape':>9}{'max |by hand - autograd|':>28}")
for name, mine, p in [("W1", gW1, model[0].weight), ("b1", gb1, model[0].bias),
                      ("W2", gW2, model[2].weight), ("b2", gb2, model[2].bias),
                      ("W3", gW3, model[4].weight), ("b3", gb3, model[4].bias)]:
    print(f"{name:>4}{str(mine.shape):>9}"
          f"{np.abs(mine - p.grad.numpy()).max():>28.2e}")

cost   by hand 2.863851   autograd 2.863851

        shape    max |by hand - autograd|
  W1   (3, 2)                    4.44e-16
  b1     (3,)                    2.22e-16
  W2   (2, 3)                    2.22e-16
  b2     (2,)                    2.22e-16
  W3   (1, 2)                    3.33e-16
  b3     (1,)                    2.22e-16


## The three rules that reproduced autograd

Twelve lines of NumPy reproduced `loss.backward()` to floating-point noise, on a
three-layer network, using nothing but:

- $\partial \mathcal{L}/\partial \mathbf{W} = \partial \mathcal{L}/\partial \mathbf{z} \cdot \mathbf{x}^{\top}$
- $\partial \mathcal{L}/\partial \mathbf{x} = \mathbf{W}^{\top} \cdot \partial \mathcal{L}/\partial \mathbf{z}$
- $\tanh'(z) = 1 - \tanh^2(z)$, which reuses the cached activation

Autograd is not doing anything you cannot do. It is doing it for an arbitrary
graph, without you writing the six lines per layer, and without getting a
transpose wrong at 2 AM.

## ANN matrix notation {.smaller}

The same network, written out in full. Worth having on hand once; not worth
reading aloud.

:::: {.columns}
::: {.column width="55%"}
[![](images/ann-matrix-notation.png){width=450 .dh460}](images/ann-matrix-notation.png){target="_blank" .zoom}
:::
::: {.column width="45%"}
Notation to keep straight:

- $\mathbf{W}^{(\ell)}$ is layer $\ell$'s weights, `(units, inputs)`
- $\mathbf{z}^{(\ell)}$ is the pre-activation
- $\mathbf{a}^{(\ell)} = g(\mathbf{z}^{(\ell)})$ is the activation
- $\mathbf{a}^{(0)} = \mathbf{x}$ is the input
- The superscript is the layer, the subscript is the unit

Every quantity in the backward pass has a matching shape on this diagram.
:::
::::

## ANN forward and backward, worked {.smaller}

:::: {.columns}
::: {.column width="50%"}
**Forward**

[![](images/ann-forward-regression.png){width=510 .dh420}](images/ann-forward-regression.png){target="_blank" .zoom}
:::
::: {.column width="50%"}
**Backward**

[![](images/ann-backward-regression.png){width=543 .dh420}](images/ann-backward-regression.png){target="_blank" .zoom}
:::
::::

Left to right, then right to left, on the same network. These are on the
handout at full size; the point here is the symmetry, not the entries.

## Jacobian memory cost

The full Jacobian of a layer holds the derivative of every output coordinate
with respect to every input coordinate. For a batch of 128, 128 features in and
256 units out, in float32:

$$128 \times 128 \times 128 \times 256 \times 4 \text{ bytes}
  \approx 6.5 \text{ GB}$$

For **one layer**. The two gradients training actually needs,
$\partial \mathcal{L}/\partial \mathbf{W}$ and
$\partial \mathcal{L}/\partial \mathbf{x}$, come to about **198 KB**.

Reverse mode never forms the Jacobian. It only ever computes
Jacobian-times-vector, which is the matrix multiply on the previous slides. That
is the difference between a model that fits in memory and one that does not.

## One entry of the weight gradient {.smaller}

[![](images/matrix-multiplication-differentiation-w11.png){width=630 .dw60}](images/matrix-multiplication-differentiation-w11.png){target="_blank" .zoom}

Expanding $\partial J / \partial w_{1,1}$ gives eight terms and **six are
zero**, leaving a column of $\mathbf{X}$ dotted with a column of
$\nabla_{\mathbf{Z}} J$. That pattern, over every entry, is exactly
$\nabla_{\mathbf{W}} J = \mathbf{X}^{\top} \nabla_{\mathbf{Z}} J$.

## Backward pass in node form: input to ReLU {.smaller}

[![](images/ann-backward-node-left.png){width=1050 .dw95}](images/ann-backward-node-left.png){target="_blank" .zoom}

One column per node, with the matrix each node hands backward written out. At
the multiply node, the two familiar rules:
$\nabla_{\mathbf{X}} J = \nabla_{\mathbf{Z}} J \mathbf{W}^{\top}$ and
$\nabla_{\mathbf{W}} J = \mathbf{X}^{\top} \nabla_{\mathbf{Z}} J$. At the
ReLU, a gate: pass the entry through where $z > 0$, otherwise zero.

## Backward pass in node form: output layer to cost {.smaller}

[![](images/ann-backward-node-right.png){width=1050 .dw95}](images/ann-backward-node-right.png){target="_blank" .zoom}

The right half of the same strip. The linear output layer has derivative 1, so
it passes its gradient straight through, and the cost node starts the whole
chain with $\nabla_{\mathbf{A}} J = \frac{2}{m}(\hat{\mathbf{Y}} -
\mathbf{Y})$.

## Exploding and vanishing gradients

[![](images/exploding-vanishing-gradient.png){width=735 .dw70}](images/exploding-vanishing-gradient.png){target="_blank" .zoom}

The gradient is a product of one factor per layer, so it compounds as it travels
right to left. Consistently large factors make it **explode** before it reaches
the early layers; consistently small ones make it **vanish**.

Either way the layers nearest the input are the ones that suffer, which is
exactly the opposite of what you want, since they set the features every later
layer builds on.

## Gradient magnitude by layer, measured {.smaller}

[![](images/fig-gradient-decay.png){width=1050 .dw85}](images/fig-gradient-decay.png){target="_blank" .zoom}

Twenty layers, real gradients read out of autograd. The chain rule
**multiplies**, so a local derivative consistently below 1 drives the product to
zero and one consistently above 1 drives it to infinity.

Sigmoid's derivative peaks at $0.25$. Twenty of those in a row is $10^{-13}$,
which is what the blue line shows: **the layers nearest the input get no signal
at all.** Scaling the weights up 60% sends the same architecture to $10^{7}$.

## Causes and cures {.smaller}

:::: {.columns .contrast}
::: {.column width="48%" .col-no}
### Exploding
Local derivatives above 1, compounding.

- Weight initialization too large, or a weight update that overshot
- **Gradient clipping** caps the norm and is very effective
- **Weight regularization**, $\mathcal{L} = \text{error} + \lambda \sum w_i^2$
- **Batch normalization**
:::
::: {.column width="48%" .col-yes}
### Vanishing
Local derivatives near 0, compounding.

- Saturating activations, and excessive depth
- **ReLU** instead of sigmoid, since its derivative is exactly 1 where it is on
- **Skip connections**, as in ResNet, hand the input forward past layers
- **Better initialization**, Kaiming or Xavier
:::
::::

This is a **training** problem, not an autodiff problem. Autograd will hand you a
gradient of $10^{15}$ without complaint. **Every cure listed here is Week 4.**

# Closing

- Lab 3
- Quiz 3 study guide
- Supplemental content
- Quiz 2, right now

## Lab 3 {.smaller}

**Due Wednesday Sep 23, 11:59 PM ET.** Moved back a week, because today went
into the fundamentals rather than the back half of this deck.

Backpropagation in NumPy, no PyTorch:

1. Work the backward pass on a small graph **by hand**, then check it against
   finite differences
2. Write a `Value` class that records its own graph, and an eight-line
   `backward()` that walks it
3. Do the same thing on matrices, for Lab 2's $1 \to 16 \to 1$ network
4. Count the forward passes both ways: **20,001 against 401**

Same rules as Labs 1 and 2: graded for **completion**, work together if you
like, solutions posted after the deadline.

::: {.callout-note}
## Why this order
Last week you paid 20,001 forward passes for 400 steps. This week you write the
thing that gets the same gradient for 401. The payoff only lands if you felt the
bill first. Parts 2 and 3 run a little past where we stopped today; next week
closes that gap well before the due date.
:::

## Quiz 3 study guide {.smaller}

**Next week, end of class.** Closed-book, no notes. A formula sheet comes with
the quiz.

It covers **what we actually did today**, which is the back half of Week 2 plus
the first half of backpropagation:

1. **Tensors**: rank, and what the axes of a batch of images or sequences mean
2. **Depth versus width**, and **universal approximation** as interpolation
   rather than extrapolation
3. **Finite differences**: $n+1$ forward passes per gradient, and why 20,000
   passes for a 49-parameter model ends the argument
4. **Epoch, iteration, batch size**, and the learning rate too large or too small
5. **Why there is no analytical solution**: too many parameters, and a
   non-convex surface full of local minima and saddle points
6. **Three ways to get a gradient**; backprop is a **subset** of automatic
   differentiation; **loss** against **cost**
7. **The chain rule**, including the multivariable form, and that
   $\nabla_{W}\mathcal{L}$ has the **same shape** as $W$
8. **Computational graphs**: why acyclic, why every node must be differentiable,
   and the **two things each node needs to know**

::: {.callout-important}
## Not on Quiz 3
The worked backward pass, summing over paths, topological order, forward against
reverse mode, the layer gradients, and PyTorch. Those wait for Quiz 4, and so
does Lab 3, which is now due after the quiz.
:::

## Supplemental content {.smaller}

These are **optional**. No graded work assumes you read them. They go deeper
than we had time for, and the first two are the source notebooks this deck was
built from.

:::: {.columns}
::: {.column width="50%"}
**Autodiff and gradients**

- [Automatic differentiation](https://jfh.georgetown.domains/centralized-lecture-content/content/machine-learning/deep-learning/backprop/automatic-differentiation/automatic-differentiation.html)
- [Matrix-multiplication differentiation](https://jfh.georgetown.domains/centralized-lecture-content/content/machine-learning/deep-learning/backprop/matrix-multiplication-differentiation/matrix-multiplication-differentiaion.html)
- [Gradient descent](https://jfh.georgetown.domains/centralized-lecture-content/content/machine-learning/deep-learning/fundamentals/gradient-descent/notes.html)
- [Backprop basics, video](https://jfh.georgetown.domains/centralized-lecture-content/content/machine-learning/deep-learning/backprop/backprop-basics-video/notes.html)
:::
::: {.column width="50%"}
**The calculus underneath**

- [Univariate differentiation, video](https://jfh.georgetown.domains/centralized-lecture-content/content/mathematics/calculus/single-variable-calculus/univariate-differentiation-fundamentals/video.html)
- [Partial derivatives](https://jfh.georgetown.domains/centralized-lecture-content/content/mathematics/calculus/multivariable-calculus/multivariable-functions-and-partial-derivatives/notes.html)
- [Calculus overview](https://jfh.georgetown.domains/centralized-lecture-content/content/mathematics/calculus/multivariable-calculus/calculus-overview/notes.html)
- [Numerical optimization](https://jfh.georgetown.domains/centralized-lecture-content/content/mathematics/solvers-and-optimization/numerical-methods/numerical-optimization/notes.html)
:::
::::

<sup>All of these live on the
[centralized lecture content](https://jfh.georgetown.domains/centralized-lecture-content/)
site. The autodiff page also carries a from-scratch `Node` class that implements
today's graph, topological sort and backward pass in about 80 lines of NumPy,
which is close to what Lab 3 asks you to write. The right-hand column is the
calculus review if the chain rule felt rusty; the PyTorch pages come with next
week's deck.</sup>

## Wrap-up {.smaller}

Where we started: 20,001 forward passes for one decent fit. Where we are:

- There is **no analytical solution** for a network's optimal weights, and the
  loss surface is non-convex, so we descend it instead of solving it
- **Finite differences** would work and cost one forward pass per parameter,
  which is the whole reason this lecture exists
- **Backpropagation** is reverse-mode autodiff applied to a network, and
  autodiff is the **superset**
- The **chain rule** is the entire mathematical content, and the multivariable
  form is the one that matters
- A **computational graph** is the forward pass, recorded. Values at the nodes,
  operations on the edges, and every node differentiable
- Each node needs exactly **two things**: the gradient handed to it from
  downstream, and the derivatives of its own inputs

The rest of this deck, from the worked backward pass onward, is where we pick up.

::: {.callout-note}
## Next week
**Finish the backward pass, then PyTorch and training.** How a node gets its
upstream gradient, what happens where paths meet, the layer-level version, and
then `.backward()` doing all of it for you.
:::

**Before next Wednesday:** Lab 3. And claim a Spotlight slot if you have not.

## Quiz 2

Closed-book, no notes. A formula sheet comes with it.

Covers **Week 2 and Lab 2**:

- Layer shapes and parameter counts
- The forward pass, single example and batched
- Why nonlinearities exist, and what stacked linear layers collapse to
- Activations: sigmoid, tanh, ReLU, softmax
- Depth versus width
- Why finite-difference gradients do not scale

Twenty-five points. Show your work; an answer with no reasoning earns partial
credit at best.